# EDA Meter Profile
# 단일 계량기 상세 EDA입니다. 
# 기본 대상은 `H1.Z16`이며, `METER_URN` 값을 바꿔 다른 계량기에도 동일하게 적용할 수 있습니다.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('/home/playdata2/final_pj/energy-platform')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.meter_metadata import get_metadata
from scripts.preprocess_h1z16 import preprocess_meter

METER_URN = 'H1.Z16'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda' / METER_URN.replace('.', '_')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = get_metadata(METER_URN)
df, df_before, issues, invalid_segments = preprocess_meter(METER_URN, print_progress=False, print_issue_details=False)
target_col = metadata['anomaly_target']
metadata

## 1. 기초 통계

In [ ]:
print('rows:', len(df))
print('valid_rows:', int(df['is_valid'].sum()))
print('issues:', len(issues))
print('invalid_segments:', len(invalid_segments))
display(df[[c for c in [target_col, 'W', 'PF', 'Ta', 'Igm'] if c in df.columns]].describe())
display(df[[c for c in [target_col, 'W', 'PF', 'Ta', 'Igm'] if c in df.columns]].isnull().sum())

## 2. 시계열 시각화

In [ ]:
cols = [c for c in [target_col, 'W', 'PF', 'Ta', 'Igm'] if c in df.columns]
fig, axes = plt.subplots(len(cols), 1, figsize=(16, 3.5 * len(cols)), sharex=True)
if len(cols) == 1:
    axes = [axes]
for ax, col in zip(axes, cols):
    ax.plot(df['ts'], df[col], linewidth=0.8)
    ax.set_title(f'{METER_URN} {col}')
plt.tight_layout()
plot_path = OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_timeseries.png'
plt.savefig(plot_path, dpi=150)
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(plot_path)))


## 3. 그룹 집계

In [ ]:
hourly = df.groupby('hour')[target_col].mean().reset_index()
monthly = df.groupby('month')[target_col].mean().reset_index()
weekend = df.groupby('is_weekend')[target_col].mean().reset_index()
display(hourly.head())
display(monthly.head())
display(weekend)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hourly['hour'], hourly[target_col], marker='o')
axes[0].set_title('Hourly Mean')
axes[1].bar(monthly['month'], monthly[target_col])
axes[1].set_title('Monthly Mean')
axes[2].bar(weekend['is_weekend'].astype(str), weekend[target_col])
axes[2].set_title('Weekend vs Weekday')
plt.tight_layout()
plot_path = OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_profiles.png'
plt.savefig(plot_path, dpi=150)
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(plot_path)))


## 4. 상관분석과 데이터 품질 분석

In [ ]:
corr_cols = [c for c in [target_col, 'PF', 'I1', 'I2', 'I3', 'P1', 'P2', 'P3', 'Ta', 'Igm'] if c in df.columns and df[c].notna().any()]
corr = df[corr_cols].corr(numeric_only=True)
display(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar()
plt.title(f'{METER_URN} Correlation Heatmap')
plt.tight_layout()
plot_path = OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_corr.png'
plt.savefig(plot_path, dpi=150)
plt.close()
from IPython.display import Image, display
display(Image(filename=str(plot_path)))

quality_df = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_ratio': df.isnull().mean(),
}).sort_values('null_ratio', ascending=False)
display(quality_df.head(15))
quality_df.to_csv(OUTPUT_DIR / f'{METER_URN.replace('.', '_')}_quality.csv', encoding='utf-8-sig')


## 5. 추가 대표 계량기 상세 EDA\n\n기존 `H1.Z16` 결과는 그대로 두고, 아래 셀에서 대표 계량기 `H1.Z20`, `V.K21`, `H2.Z61`을 같은 방식으로 추가 분석합니다.

In [ ]:
REPRESENTATIVE_METERS = ['H1.Z20', 'V.K21', 'H2.Z61']
rep_summary_rows = []
from IPython.display import Image, display

for meter in REPRESENTATIVE_METERS:
    meta = get_metadata(meter)
    rep_df, rep_df_before, rep_issues, rep_invalid_segments = preprocess_meter(meter, print_progress=False, print_issue_details=False)
    rep_target = meta['anomaly_target']
    rep_output_dir = PROJECT_ROOT / 'outputs' / 'eda' / meter.replace('.', '_')
    rep_output_dir.mkdir(parents=True, exist_ok=True)

    rep_summary_rows.append({
        'meter_urn': meter,
        'meter_type': meta['meter_type'],
        'group_name': meta['group_name'],
        'anomaly_target': rep_target,
        'rows': len(rep_df),
        'valid_rows': int(rep_df['is_valid'].sum()),
        'issue_count': len(rep_issues),
        'invalid_segments': len(rep_invalid_segments),
    })

    rep_cols = [c for c in [rep_target, 'W', 'PF', 'Ta', 'Igm', 'qv'] if c in rep_df.columns]
    fig, axes = plt.subplots(len(rep_cols), 1, figsize=(16, 3.5 * len(rep_cols)), sharex=True)
    if len(rep_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, rep_cols):
        ax.plot(rep_df['ts'], rep_df[col], linewidth=0.8)
        ax.set_title(f'{meter} {col}')
    plt.tight_layout()
    ts_path = rep_output_dir / f'{meter.replace('.', '_')}_timeseries.png'
    plt.savefig(ts_path, dpi=150)
    plt.close(fig)
    display(Image(filename=str(ts_path)))

    hourly = rep_df.groupby('hour')[rep_target].mean().reset_index()
    monthly = rep_df.groupby('month')[rep_target].mean().reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hourly['hour'], hourly[rep_target], marker='o')
    axes[0].set_title(f'{meter} Hourly Mean')
    axes[1].bar(monthly['month'], monthly[rep_target])
    axes[1].set_title(f'{meter} Monthly Mean')
    plt.tight_layout()
    pr_path = rep_output_dir / f'{meter.replace('.', '_')}_profiles.png'
    plt.savefig(pr_path, dpi=150)
    plt.close(fig)
    display(Image(filename=str(pr_path)))

    corr_cols = [c for c in [rep_target, 'PF', 'I1', 'I2', 'I3', 'P1', 'P2', 'P3', 'Ta', 'Igm', 'qv', 'Tvl', 'Trl'] if c in rep_df.columns and rep_df[c].notna().any()]
    if len(corr_cols) >= 2:
        corr = rep_df[corr_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        plt.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
        plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
        plt.yticks(range(len(corr_cols)), corr_cols)
        plt.colorbar()
        plt.title(f'{meter} Correlation Heatmap')
        plt.tight_layout()
        corr_path = rep_output_dir / f'{meter.replace('.', '_')}_corr.png'
        plt.savefig(corr_path, dpi=150)
        plt.close()
        display(Image(filename=str(corr_path)))

    quality = pd.DataFrame({'null_count': rep_df.isnull().sum(), 'null_ratio': rep_df.isnull().mean()}).sort_values('null_ratio', ascending=False)
    quality.to_csv(rep_output_dir / f'{meter.replace('.', '_')}_quality.csv', encoding='utf-8-sig')

rep_summary_df = pd.DataFrame(rep_summary_rows)
display(rep_summary_df)
rep_summary_df.to_csv(PROJECT_ROOT / 'outputs' / 'eda' / 'representative_meter_profile_summary.csv', index=False, encoding='utf-8-sig')
print('saved:', PROJECT_ROOT / 'outputs' / 'eda' / 'representative_meter_profile_summary.csv')
